# KAVI audience enrichment

Answers the finder's question *"Who is it for?"* for every resource. Run the cells from top to bottom.

Reads `data/cleaned/resources_extended.csv` and the PDF listing text in `data/extracted/connections_pdf_listings.csv`. Writes:

- `data/cleaned/resource_audiences.csv` - one row per `resource_id`, to be joined onto the resource table
- `data/review/audience_review.csv` - a random sample for a manual check of the rules
- `data/review/audience_no_match.csv` - rows the patterns could not match or confirm, for a manual look

## Rules

1. **`pdf_audiences` stays as it is.** It comes from the PDF's audience chapters and is never changed or removed.
2. **Confirmation:** a PDF audience is *confirmed* when the organization's name, its description or its PDF listing text also mentions it.
3. **Audiences from the text** are added, including two the PDF has no chapter for (*Survivors of violence*, *People without homes*).
4. **`Everyone` is the default.** A resource is *restricted* only when the text says so explicitly ("only", "must be", an age range). A resource that merely focuses on a group keeps `Everyone` and gets the group as an extra tag.
5. **No match, no restriction:** when the texts don't confirm a row's audiences (a PDF audience no text mentions, or a restriction with no audience named anywhere), the audiences are kept from wherever they came from, and `Everyone` is always added.
6. **Ranking:** when a user picks an audience, e.g. *Youth*, resources focused on youth come first, then other resources tagged *Youth*, then *Everyone*. Section 5 has the function.

In [ ]:
import os
import re

import pandas as pd

IN_FILE = "../../data/cleaned/resources_extended.csv"
PDF_FILE = "../../data/extracted/connections_pdf_listings.csv"
OUT_FILE = "../../data/cleaned/resource_audiences.csv"
REVIEW_FILE = "../../data/review/audience_review.csv"
NO_MATCH_FILE = "../../data/review/audience_no_match.csv"
REVIEWED_FILE = "reviewed_audiences.csv"  # your manual decisions; never overwritten by this notebook
os.makedirs(os.path.dirname(REVIEW_FILE), exist_ok=True)

df = pd.read_csv(IN_FILE, dtype=str, keep_default_na=False)
pdf = pd.read_csv(PDF_FILE, dtype=str, keep_default_na=False)
df["pdf_text"] = df["pdf_listing_id"].map(pdf.set_index("listing_id")["listing_text"]).fillna("")

# Some descriptions still hold notes from the PDF extraction, e.g. "[PARTIAL/POINTER ONLY — … see the
# LGBTQI+ chapter …]". They name chapters, not who a resource serves, so they are removed before matching.
EXTRACTION_NOTE = r"\[[^\]]*(?:PARTIAL|POINTER|INCOMPLETE|CORRECTED|NOTE|segment)[^\]]*\]?"
for column in ("description", "pdf_text"):
    df[column] = df[column].str.replace(EXTRACTION_NOTE, " ", regex=True, flags=re.I).str.strip()

pd.set_option("display.max_colwidth", 200)
print(len(df), "rows,", (df["pdf_audiences"] != "").sum(), "with pdf_audiences,", (df["pdf_text"] != "").sum(), "with PDF listing text")

## 1. Audience patterns

One regular expression per audience, matched case-insensitively. The names of the first nine are the PDF's audience chapters, so they line up with `pdf_audiences`.

Each pattern is run separately on three texts, so it is visible where a match came from:

- `name_audiences` - the organization's name (e.g. "… Center **for Women**")
- `description_audiences` - the description
- `pdf_text_audiences` - the listing's text in the PDF

In [ ]:
AUDIENCES = {
    "Youth": r"\byouth|young (people|adults|men|women)|\bteens?\b|teenagers?|adolescents?|\bminors?\b",
    "Women": r"\bwomen\b|\bgirls?\b|\bmothers?\b",
    "LGBTQI+": r"lgbt|transgender|\btrans\b|queer|gender[- ](non-?conforming|fluid|expansive)|two[- ]spirit",
    "Veterans": r"veterans?\b|service members|military",
    "Immigrants": r"immigra|undocumented|asylum|refugees?\b|migrants?\b|new americans",
    "Older People": r"\bseniors?\b|older (adults|people|new yorkers)|elderly|\baging\b|ages? (6[0-9]|5[5-9]) (and|or) (older|over|up)",
    "People with Disabilities": r"disabilit|\bdeaf\b|\bblind\b|hard of hearing|visually impaired|independent living",
    "Parents & Caregivers": r"\b(grand)?parents?\b|caregivers?\b|guardians?\b|families with children|children of incarcerated|parenting",
    "Formerly Incarcerated People": r"formerly incarcerated|justice[- ](involved|impacted)|re-?entry(?! permits?)|returning (citizens|home)|(currently|recently) incarcerated|(leaving|released from|after) (prison|jail)|on (parole|probation)|(impacted|affected) by (incarceration|the (criminal )?(legal|justice) system)",
    # "survivors" alone also means a veteran's surviving family or Holocaust survivors, so it needs a violence word.
    "Survivors of violence": r"domestic violence|intimate partner|dating (abuse|violence)|sexual (assault|abuse|violence|trauma)|gender[- ]based violence|trafficking|survivors? of (\w+ ){0,3}(violence|abuse|assault|crime|trafficking|rape|trauma)|crime victims?|victims? of (crime|violence|abuse)",
    "People without homes": r"homeless|unhoused|without homes|street homeless",
}

def found_audiences(text):
    return [name for name, pattern in AUDIENCES.items() if re.search(pattern, text, re.I)]

def split(value):
    return [v.strip() for v in value.split(";") if v.strip()]

def join(values):
    return "; ".join(dict.fromkeys(values))

for column, text in [("name_audiences", "org_name"), ("description_audiences", "description"), ("pdf_text_audiences", "pdf_text")]:
    df[column] = df[text].apply(lambda t: join(found_audiences(t)))
df["text_audiences"] = df.apply(lambda r: join(split(r["name_audiences"]) + split(r["description_audiences"]) + split(r["pdf_text_audiences"])), axis=1)

## 2. Confirm the PDF audiences

A PDF audience is **confirmed** when the name, the description or the PDF listing text also mentions it. An unconfirmed one stays in the data (rule 1), but is listed for review: often the organization serves that group without saying so, sometimes the PDF placed it loosely, and sometimes a pattern above is missing a word.

`pdf_audience_check` summarises each row: `confirmed`, `partly confirmed`, `not confirmed` or blank (no PDF audience).

In [ ]:
def confirm(row):
    pdf_aud, text_aud = split(row["pdf_audiences"]), split(row["text_audiences"])
    confirmed = [a for a in pdf_aud if a in text_aud]
    unconfirmed = [a for a in pdf_aud if a not in text_aud]
    if not pdf_aud:
        check = ""
    elif not unconfirmed:
        check = "confirmed"
    elif confirmed:
        check = "partly confirmed"
    else:
        check = "not confirmed"
    return pd.Series({"confirmed_pdf_audiences": join(confirmed), "unconfirmed_pdf_audiences": join(unconfirmed), "pdf_audience_check": check})

df = df.join(df.apply(confirm, axis=1))
df["pdf_audience_check"].replace("", "(no PDF audience)").value_counts()

## 3. Restricted, focused, or for everyone

**Restricted is deliberately strict**, so that no resource is hidden from people it would actually serve. A resource is restricted to an audience only when the description or PDF text says so explicitly:

- an exclusivity word next to the audience: *only*, *exclusively*, *solely*, *limited to*, *restricted to*, *must be* (e.g. "open **only to** veterans", "**must be** a woman", "for youth **only**"), or
- an age range in the **first sentence**, where it defines the whole program: *ages 13–24*, *18 to 24 years old*. A range from 12 up to at most 25 counts as *Youth*, a lower limit of 55 or above as *Older People*. Ranges starting below 12 (e.g. *ages 0–5*) are children's programs that parents use, so they don't restrict. An age range later in the text usually describes one sub-program, so it doesn't restrict either.

Only spaces may sit between the exclusivity word and the audience, so "(collect calls only) … transgender" or "… only. Women" don't count.

Everything weaker is a **focus**, not a restriction. A focus keeps `Everyone`, but ranks the resource higher for that audience:

- the audience is in the organization's name ("Center for **Women**"), or
- the first sentence says the service is *for* / *serves* / *supports* that group ("Provides programs **for** LGBTQIA+ **youth**").

`audience_restriction` is `restricted`, `open` (says it serves anyone), `mixed` (both; needs review) or `not_stated`. Every resource that isn't `restricted` gets `Everyone`. `restriction_evidence` keeps the exact words that triggered a restriction or the open wording.

In [ ]:
WINDOW = r"(?:[ \t]+[\w'’-]+){0,5}?[ \t]+"  # words separated by spaces only, so ")", "." or "," break a match
EXCLUSIVE = r"\b(only|exclusively|solely|limited to|restricted to|must be)\b"
AGE_RANGE = r"\bages? (\d{1,2})\s*(?:-|–|—|to|through)\s*(\d{1,2})\b|\b(\d{1,2})\s*(?:-|–|—|to)\s*(\d{1,2})\s*years? old\b"
OPEN = r"\b(anyone|everyone|all new yorkers|all (?:ages|people|residents|individuals)|regardless of|open to (?:all|the public)|no (?:eligibility|referral) (?:requirements?|needed))\b"
CUE = r"\b(for|serv(?:es|ing)|supports?|assists?|helps?|connects|works with|dedicated to|designed for|run by and for)\b"

def first_sentence(text):
    return re.split(r"(?<=[.!?])\s", text.strip(), maxsplit=1)[0]

def explicit_restrictions(text, first_sentences):
    found = {}
    for name, pattern in AUDIENCES.items():
        aud = "(?:" + pattern + ")"
        m = re.search(EXCLUSIVE + WINDOW + aud, text, re.I) or re.search(aud + r"(?:[ \t]+[\w'’-]+){0,2}?[ \t]+only\b", text, re.I)
        if m:
            found[name] = m.group(0)
    for sentence in first_sentences:
        for m in re.finditer(AGE_RANGE, sentence, re.I):
            low, high = [int(x) for x in m.groups() if x][:2]
            if 12 <= low and high <= 25:
                found.setdefault("Youth", m.group(0))
            elif low >= 55:
                found.setdefault("Older People", m.group(0))
    return found

def focus(row):
    named = split(row["name_audiences"])
    first = first_sentence(row["description"])
    cued = [n for n, p in AUDIENCES.items() if re.search(CUE + r"(?:\W+[\w'’-]+){0,6}?\W+(?:" + p + ")", first, re.I)]
    return join(named + cued)

def classify(row):
    text = row["description"] + " " + row["pdf_text"]
    limits = explicit_restrictions(text, [first_sentence(row["description"]), first_sentence(row["pdf_text"])])
    is_open = re.search(OPEN, text, re.I)
    if limits and is_open:
        status = "mixed"
    elif limits:
        status = "restricted"
    elif is_open:
        status = "open"
    else:
        status = "not_stated"
    evidence = list(limits.values()) + ([is_open.group(0)] if is_open else [])
    return pd.Series({
        "restricted_audiences": join(limits),
        "focus_audiences": focus(row),
        "audience_restriction": status,
        "restriction_evidence": " | ".join(evidence),
        "everyone_basis": {"restricted": "", "mixed": "stated", "open": "stated", "not_stated": "assumed"}[status],
    })

df = df.join(df.apply(classify, axis=1))
df["audience_restriction"].value_counts()

## 4. Combine

First the **no-match rule**: a row whose audiences the texts don't confirm is never restricted. The restriction becomes a focus (so it still ranks first for that audience) and the row gets `Everyone`.

`audiences_all` is what the finder filters on: `pdf_audiences` first (unchanged), then the audiences found in the name, description and PDF text, then `Everyone` for every resource that isn't restricted.

In [ ]:
# No-match rule: a row whose audiences the texts don't confirm is never restricted. Its audiences
# stay (from the PDF or the text), the restriction becomes a focus, and it gets Everyone.
no_match = (df["unconfirmed_pdf_audiences"] != "") | ((df["text_audiences"] == "") & (df["audience_restriction"] == "restricted"))
lifted = no_match & (df["audience_restriction"] == "restricted")
df.loc[lifted, "focus_audiences"] = df[lifted].apply(lambda r: join(split(r["focus_audiences"]) + split(r["restricted_audiences"])), axis=1)
df.loc[lifted, "restricted_audiences"] = ""
df.loc[lifted, "audience_restriction"] = "not_stated"
df.loc[lifted, "everyone_basis"] = "assumed"
print(lifted.sum(), "restricted rows without confirmed audiences now get Everyone")

def combine(row):
    tags = split(row["pdf_audiences"]) + split(row["text_audiences"]) + split(row["restricted_audiences"]) + split(row["focus_audiences"])
    if row["everyone_basis"]:
        tags.append("Everyone")
    return join(tags)

df["audiences_all"] = df.apply(combine, axis=1)
df["audience_needs_review"] = (
    (df["audience_restriction"].isin(["restricted", "mixed"]))
    | (df["pdf_audience_check"].isin(["partly confirmed", "not confirmed"]))
)

print("Everyone:", df["audiences_all"].str.contains("Everyone").sum(), "of", len(df))
print("Everyone plus a specific audience:", (df["audiences_all"].str.contains("Everyone") & (df["audiences_all"] != "Everyone")).sum())
print("Restricted:", (df["audience_restriction"] == "restricted").sum())
df[df["audience_restriction"] == "restricted"][["org_name", "restricted_audiences", "restriction_evidence"]]

## 5. Ranking for the finder

When a user picks an audience, results come in this order:

1. **Made for that audience**: restricted to it, or focused on it (name or first sentence)
2. **Tagged with that audience**: from the PDF or a mention in the text
3. **Everyone**: not restricted, no specific mention

Resources restricted to *other* groups are left out. Because restricted is so strict, that is a small number. Picking `Everyone`, or nothing, shows every resource that isn't restricted.

The same logic has to go into the finder's filter (`finder/filter.js` in #16). This function is the reference for it.

In [ ]:
def rank_for(data, audience):
    """Resources for one audience, best matches first. Adds a `rank` column (1, 2 or 3)."""
    def rank(row):
        if audience in split(row["restricted_audiences"]) or audience in split(row["focus_audiences"]):
            return 1
        if audience in split(row["audiences_all"]):
            return 2
        if row["everyone_basis"]:
            return 3
        return None
    ranked = data.assign(rank=data.apply(rank, axis=1)).dropna(subset=["rank"])
    return ranked.astype({"rank": int}).sort_values(["rank", "org_name"])

youth = rank_for(df, "Youth")
print(youth["rank"].value_counts().sort_index().rename({1: "1 made for Youth", 2: "2 tagged Youth", 3: "3 Everyone"}))
youth[["rank", "org_name", "audiences_all", "focus_audiences"]].head(15)

## 6. Rows to look at

Written to `audience_no_match.csv`, one row per issue:

- `PDF audience not confirmed`: the PDF puts the organization in an audience chapter, but neither its name, description nor PDF text mentions that audience. Often a missing word in the patterns.
- `no audience anywhere`: neither the PDF nor any text names an audience. These get only `Everyone`, which is usually right; a quick look finds missed patterns.

Cross-reference rows (`row_type = cross_reference`) only say "See the full listing …", so they rarely match. They are sorted to the end.

In [ ]:
issues = pd.concat([
    df[df["unconfirmed_pdf_audiences"] != ""].assign(issue="PDF audience not confirmed"),
    df[(df["pdf_audiences"] == "") & (df["text_audiences"] == "")].assign(issue="no audience anywhere"),
])
issues["is_cross_reference"] = issues["row_type"] == "cross_reference"
issues = issues.sort_values(["is_cross_reference", "issue", "org_name"])

print(issues.groupby(["is_cross_reference", "issue"]).size())
issues[["resource_id", "org_name", "row_type", "issue", "pdf_audiences", "unconfirmed_pdf_audiences",
        "description", "pdf_text"]].to_csv(NO_MATCH_FILE, index=False)
print(len(issues), "rows written to", NO_MATCH_FILE)

issues[(issues["issue"] == "PDF audience not confirmed") & ~issues["is_cross_reference"]][["org_name", "unconfirmed_pdf_audiences", "description"]].head(20)

## 7. Apply manual decisions

`reviewed_audiences.csv` (next to this notebook) holds decisions a person made for single rows, where no pattern can capture the reason. It is read on every run and overrides the rules above. Nothing in this notebook ever writes to it, except creating it empty the first time.

| Column | Meaning |
|---|---|
| `resource_id`, `org_name` | The row. The notebook stops if the name no longer matches the id. |
| `human_audiences` | The correct audiences, separated by `; `, without `Everyone` (it is added automatically unless the row is restricted). Blank: keep the rule-based ones. |
| `human_restriction` | `restricted`, `open` or `not_stated`. Blank: keep the rule-based value. |
| `human_notes` | Why. |

To add a decision, copy the row's `resource_id` and `org_name` from `audience_review.csv` or `audience_no_match.csv` into this file and fill in the other columns.

In [ ]:
if not os.path.exists(REVIEWED_FILE):
    pd.DataFrame(columns=["resource_id", "org_name", "human_audiences", "human_restriction", "human_notes"]).to_csv(REVIEWED_FILE, index=False)
reviewed = pd.read_csv(REVIEWED_FILE, dtype=str, keep_default_na=False)

df["audience_reviewed"] = False
for _, r in reviewed.iterrows():
    rid = r["resource_id"].strip()
    match = df.index[df["resource_id"] == rid]
    if len(match) == 0:
        raise ValueError(f"reviewed_audiences.csv: resource_id {rid} is not in the data")
    i = match[0]
    if df.at[i, "org_name"].strip().lower() != r["org_name"].strip().lower():
        raise ValueError(f"reviewed_audiences.csv: resource_id {rid} is '{df.at[i, 'org_name']}', not '{r['org_name']}'")
    status = r["human_restriction"].strip()
    if status:
        if status not in ("restricted", "open", "not_stated"):
            raise ValueError(f"reviewed_audiences.csv: resource_id {rid}: human_restriction must be restricted, open or not_stated")
        df.at[i, "audience_restriction"] = status
        df.at[i, "everyone_basis"] = {"restricted": "", "open": "stated", "not_stated": "assumed"}[status]
    audiences = [a for a in split(r["human_audiences"]) if a != "Everyone"] or [a for a in split(df.at[i, "audiences_all"]) if a != "Everyone"]
    if df.at[i, "audience_restriction"] == "restricted" and r["human_audiences"].strip():
        df.at[i, "restricted_audiences"] = join(audiences)
    df.at[i, "audiences_all"] = join(audiences + (["Everyone"] if df.at[i, "everyone_basis"] else []))
    df.at[i, "audience_reviewed"] = True
    df.at[i, "audience_needs_review"] = False

print(len(reviewed), "manual decisions applied")

## 8. Save, plus a review sample

`audience_review.csv` is a random sample of 50 rows with empty columns for a person's verdict. Check it, then sharpen the patterns above and run again.

In [ ]:
cols = ["resource_id", "org_name", "pdf_audiences", "confirmed_pdf_audiences", "unconfirmed_pdf_audiences",
        "pdf_audience_check", "name_audiences", "description_audiences", "pdf_text_audiences",
        "restricted_audiences", "focus_audiences", "audience_restriction", "restriction_evidence",
        "everyone_basis", "audiences_all", "audience_needs_review", "audience_reviewed"]
df[cols].to_csv(OUT_FILE, index=False)

sample = df[(df["row_type"] != "cross_reference") & ~df["audience_reviewed"]].sample(50, random_state=42)
review = sample[["resource_id", "org_name", "description", "pdf_audiences", "pdf_audience_check",
                 "restricted_audiences", "focus_audiences", "audience_restriction", "audiences_all"]].copy()
review["human_verdict"] = ""        # agree / change / unsure
review["human_audiences"] = ""      # only for change: the correct list, separated by "; "
review["human_restriction"] = ""    # only for change: restricted / open / not_stated
review["human_notes"] = ""
review.sort_values("resource_id", key=lambda s: s.astype(int)).to_csv(REVIEW_FILE, index=False)

print(len(df), "rows written to", OUT_FILE)
print(len(review), "rows written to", REVIEW_FILE)